# Skin lesion boundary detection — complete guided lab

**Goal:** Find a skin lesion's outline in **five** images, then estimate the number of enclosed pixels and the outline length. This is image processing; **no model training is required**.

Run cells from top to bottom in Google Colab. Each code cell has an explanation immediately above it. The dataset is **HAM10000**, as in your Lab 2 and Lab 3. `kagglehub` makes it available automatically in a fresh Colab session; you do not manually download the dataset. Images and results remain in `/content/lesion_boundary_results` until you download the final ZIP.

**Honesty rule:** Without expert-drawn masks, we cannot claim that an automatically selected contour is the true medical lesion boundary. Inspect the overlays before submitting. A pixel area is not an area in mm² without a calibrated pixel-to-mm scale.

## 1. Learn the ideas before the experiment

| Concept | Very simple meaning | Tiny example |
|---|---|---|
| Pixel | One tiny square of an image | A grayscale pixel can be 0 (black), 128 (gray), or 255 (white). |
| RGB image | Each pixel has red, green and blue values | `(210, 160, 150)` describes one skin-colored pixel. |
| Grayscale | One brightness number per pixel | Converting RGB makes brightness edges easier to study. |
| Kernel / filter | A small grid moved across the image | A 3×3 mean kernel averages 9 nearby pixels. |
| Gaussian blur | Weighted average, strongest near center | Nearby pixels 20, 20, 200 can be softened before edge detection. |
| Edge | A strong change between neighboring pixels | `30, 32, 35, 180, 185`: the big jump suggests a boundary. |
| Sobel | Estimates brightness changes in two directions | Strong `Gx` highlights a vertical boundary. |
| Canny | Keeps thin edges using two thresholds and connection tracking | A weak pixel stays if it connects to a strong edge. |
| Contour | Ordered points following an outline | The perimeter of a drawn shape. |
| Morphological closing | Connects small breaks in white edges | Turns an almost closed ring into a more continuous ring. |
| Mask | White = proposed lesion, black = background | Count white pixels to get approximate area. |

**Canny thresholds:** `(50, 100)` means a lower and an upper threshold, not 50 to 100 pixels of area. Higher thresholds usually give fewer edges. Low thresholds can also keep noise. Canny uses a gradient and non-maximum suppression, then links weak edges to strong edges. Blur reduces tiny changes before this step.

**Area versus perimeter:** Area counts pixels **inside** a filled contour. Perimeter measures the **length around** the contour in pixels. A 10×10 filled square has roughly 100 pixel positions inside, while each of its four sides is about 10 pixels long. The two measures have different units.

**Workflow:** original → grayscale → Gaussian blur → three Canny maps → join small edge gaps → find plausible contours → choose and review → draw boundary → area/perimeter → compare eight methods.

## 2. Set up Colab

This cell imports only the tools we use. `cv2` is OpenCV for filters, Canny and contours; NumPy stores image arrays; pandas makes tables; Matplotlib shows images. The notebook installs `kagglehub` for the same automatic HAM10000 access used previously.

In [ ]:
%pip -q install kagglehub

from pathlib import Path
import random
import zipfile
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display
import kagglehub

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
OUT = Path('/content/lesion_boundary_results')
OUT.mkdir(parents=True, exist_ok=True)
print('OpenCV:', cv2.__version__, '| output:', OUT)

## 3. See how a small filter works

We use a 3×3 patch to see an average filter in numbers. The center starts at 200, while eight neighbors equal 20; their average is much lower. Gaussian uses **weighted** averaging instead of treating all nine equally.

In [ ]:
patch = np.array([[20, 20, 20], [20, 200, 20], [20, 20, 20]], dtype=np.uint8)
mean_value = patch.mean()
gaussian_value = cv2.GaussianBlur(patch, (3, 3), 0)[1, 1]
print('3×3 patch:\n', patch)
print('Mean of all nine numbers:', round(float(mean_value), 2))
print('Gaussian-filtered center (OpenCV uses border padding):', int(gaussian_value))

## 4. Select five actual HAM10000 images

The code finds the metadata CSV and image files, then picks one reproducible example from each of five diagnostic categories. The categories are **dataset labels**, not predictions from our boundary detector. Run the first time with `IMAGE_IDS = []`. If a selected photo has a ruler, hair, or a cropped lesion, change its ID after viewing the sample sheet, then rerun from here. No Lab 1/2 training split is required for a five-image boundary exercise.

In [ ]:
DATA_ROOT = Path(kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000'))
PREFERRED_CLASSES = ['mel', 'bcc', 'bkl', 'nv', 'akiec', 'df', 'vasc']
IMAGE_IDS = []  # Optional: enter exactly five HAM10000 image IDs for a reproducible manual selection.

metadata_files = list(DATA_ROOT.rglob('HAM10000_metadata.csv'))
assert metadata_files, 'HAM10000_metadata.csv was not found.'
meta = pd.read_csv(metadata_files[0])
image_paths = {p.stem: p for p in DATA_ROOT.rglob('*.jpg')}
meta['path'] = meta['image_id'].map(image_paths)
meta = meta.dropna(subset=['path']).drop_duplicates('image_id')

if IMAGE_IDS:
    assert len(IMAGE_IDS) == 5 and len(set(IMAGE_IDS)) == 5, 'Enter five distinct image IDs.'
    chosen = meta.set_index('image_id').loc[IMAGE_IDS].reset_index()
else:
    # Look through a small, reproducible group of photos in each class.
    # Prefer a centered, reasonably compact edge ring, so hair/ruler lines are less likely.
    def sample_quality(path):
        g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if g is None: return -1
        blur = cv2.GaussianBlur(g, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 100)
        connected = cv2.morphologyEx(edges, cv2.MORPH_CLOSE,
                    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=2)
        contours, _ = cv2.findContours(connected, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        h, w = g.shape
        scores = []
        for c in contours:
            area, per = cv2.contourArea(c), cv2.arcLength(c, True)
            x, y, cw, ch = cv2.boundingRect(c)
            if per == 0 or not (.015 <= area/(h*w) <= .45): continue
            if min(x, y) < 4 or x+cw >= w-4 or y+ch >= h-4: continue
            if area/(cw*ch) < .50 or 4*np.pi*area/(per*per) < .55: continue
            m = cv2.moments(c)
            if m['m00'] == 0: continue
            dist = np.hypot((m['m10']/m['m00']-w/2)/w,
                            (m['m01']/m['m00']-h/2)/h)
            if dist > .26: continue
            scores.append(area/(h*w) + .1*(1-dist/.26))
        return max(scores, default=-1)

    selected_rows = []
    for i, label in enumerate(PREFERRED_CLASSES):
        options = meta[meta['dx'].eq(label)].sample(
            n=min(20, int(meta['dx'].eq(label).sum())), random_state=SEED+i)
        scores = [(sample_quality(row['path']), row) for _, row in options.iterrows()]
        quality, selected = max(scores, key=lambda pair: pair[0])
        if quality >= 0:
            selected_rows.append(selected)
        if len(selected_rows) == 5: break
    assert len(selected_rows) == 5, 'Could not find five plausible example photos. Set IMAGE_IDS manually.'
    chosen = pd.DataFrame(selected_rows).reset_index(drop=True)

records = []
for _, row in chosen.iterrows():
    bgr = cv2.imread(str(row['path']))
    assert bgr is not None, f"Could not read {row['path']}"
    records.append({'id': row['image_id'], 'class': row['dx'],
                    'rgb': cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)})
print('Five selected images (same HAM10000 source as the earlier labs):')
display(chosen[['image_id', 'dx']].reset_index(drop=True))
fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, r in zip(axes, records):
    ax.imshow(r['rgb']); ax.set_title(f"{r['id']}\n{r['class']}"); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'selected_images.png', dpi=150); plt.show()

## 5. Grayscale and Gaussian preprocessing

Grayscale removes color channels. A 5×5 Gaussian filter reduces isolated small brightness changes, which can otherwise look like false edges. This cell displays all five originals alongside their grayscale and blurred versions.

In [ ]:
for r in records:
    r['gray'] = cv2.cvtColor(r['rgb'], cv2.COLOR_RGB2GRAY)
    r['gaussian'] = cv2.GaussianBlur(r['gray'], (5, 5), 0)

fig, axes = plt.subplots(5, 3, figsize=(11, 17))
for i, r in enumerate(records):
    for ax, im, name in zip(axes[i], [r['rgb'], r['gray'], r['gaussian']],
                            ['Original', 'Grayscale', 'Gaussian 5×5']):
        ax.imshow(im, cmap=None if im.ndim == 3 else 'gray', vmin=None if im.ndim == 3 else 0,
                  vmax=None if im.ndim == 3 else 255)
        ax.set_title(f"{r['id']} — {name}"); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'step1_preprocessing.png', dpi=150); plt.show()

## 6. Compare three Canny threshold pairs

The first number is the **low** threshold; the second is the **high** threshold. White pixels are proposed edges. View all three settings for every image and watch for broken lesion outlines or extra edges from hair and texture. This is the lab's Task 3.

In [ ]:
CANNY_SETTINGS = [(50, 100), (100, 200), (150, 250)]
for r in records:
    r['canny'] = {setting: cv2.Canny(r['gaussian'], *setting)
                  for setting in CANNY_SETTINGS}

fig, axes = plt.subplots(5, 4, figsize=(13, 16))
for i, r in enumerate(records):
    axes[i, 0].imshow(r['rgb']); axes[i, 0].set_title(f"{r['id']} original")
    for j, setting in enumerate(CANNY_SETTINGS, 1):
        axes[i, j].imshow(r['canny'][setting], cmap='gray', vmin=0, vmax=255)
        axes[i, j].set_title(f'Canny {setting[0]}–{setting[1]}')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'step2_three_canny_thresholds.png', dpi=150); plt.show()

## 7. Turn edges into candidate boundaries

A Canny map is a set of white lines, **not** a filled lesion. Morphological closing can join short gaps; `findContours` traces closed shapes; filters reject tiny shapes, shapes touching the image border, and shapes far from the center. The score favors a sizeable centered contour and a brightness difference between its interior and nearby skin. These are heuristics, not proof that the contour is medically correct. If no plausible contour appears, the code returns `None` rather than inventing a result.

In [ ]:
def find_boundary(edges, gray):
    h, w = gray.shape
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, close_kernel, iterations=2)
    contours, _ = cv2.findContours(closed, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    for c in contours:
        contour_area = cv2.contourArea(c)
        fraction = contour_area / (h * w)
        x, y, cw, ch = cv2.boundingRect(c)
        if not (0.006 <= fraction <= 0.55):
            continue
        perimeter = float(cv2.arcLength(c, True))
        compactness = 4*np.pi*contour_area/(perimeter*perimeter) if perimeter else 0
        if contour_area/(cw*ch) < .32 or compactness < .24:
            continue  # likely a long hair/ruler line rather than the lesion
        if x <= 2 or y <= 2 or x + cw >= w - 2 or y + ch >= h - 2:
            continue
        moment = cv2.moments(c)
        if moment['m00'] == 0: continue
        cx, cy = moment['m10'] / moment['m00'], moment['m01'] / moment['m00']
        distance = np.hypot((cx-w/2)/w, (cy-h/2)/h)
        if distance > 0.34: continue

        mask = np.zeros((h, w), np.uint8)
        cv2.drawContours(mask, [c], -1, 255, thickness=cv2.FILLED)
        ring_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
        ring = cv2.dilate(mask, ring_kernel) > 0
        ring &= mask == 0
        if not np.any(ring): continue
        inside = float(np.median(gray[mask > 0]))
        outside = float(np.median(gray[ring]))
        contrast = abs(outside - inside)
        # Approximate ranking only; a human must inspect the overlay.
        score = 1.8*min(fraction/0.10, 1.2) + 1.5*(1-distance/0.34) + min(contrast/40, 1)
        candidates.append({'contour': c, 'mask': mask, 'score': score,
                           'contrast': contrast, 'area': int(np.count_nonzero(mask)),
                           'perimeter': perimeter,
                           'area_fraction': fraction})
    candidates.sort(key=lambda d: d['score'], reverse=True)
    return candidates, closed

print('Boundary finder ready. It measures mask pixel count and contour arc length.')

## 8. Inspect candidates and select the clearest setting

For each photo this cell shows the **best candidate from each Canny pair**, traced in yellow. An automatic score proposes one threshold, then you inspect it. To change a result, write that image ID and a threshold tuple in `THRESHOLD_OVERRIDES`; for a different contour under the same threshold, use `CONTOUR_RANK_OVERRIDES` (0 = best score, 1 = next). Rerun from this cell after editing. If there is no good outline in any panel, use a better lesion photo rather than reporting a false area.

In [ ]:
THRESHOLD_OVERRIDES = {}       # Example: {'ISIC_0024310': (50, 100)}
CONTOUR_RANK_OVERRIDES = {}    # Example: {'ISIC_0024310': 1}

fig, axes = plt.subplots(5, 3, figsize=(13, 17))
for i, r in enumerate(records):
    r['candidates'] = {}
    for j, setting in enumerate(CANNY_SETTINGS):
        candidates, closed = find_boundary(r['canny'][setting], r['gaussian'])
        r['candidates'][setting] = candidates
        overlay = r['rgb'].copy()
        if candidates:
            cv2.drawContours(overlay, [candidates[0]['contour']], -1, (255, 255, 0), 3)
        axes[i,j].imshow(overlay)
        axes[i,j].set_title(f"{r['id']} | {setting[0]}–{setting[1]} | " +
                            (f"score {candidates[0]['score']:.2f}" if candidates else 'NO CANDIDATE'),
                            fontsize=10)
        axes[i,j].axis('off')
plt.tight_layout(); plt.savefig(OUT/'step3_candidate_boundaries.png', dpi=150); plt.show()

for r in records:
    available = [(setting, items[0]['score']) for setting, items in r['candidates'].items() if items]
    assert available, f"No plausible contour for {r['id']}. Choose another photo and rerun."
    auto_setting = max(available, key=lambda pair: pair[1])[0]
    setting = THRESHOLD_OVERRIDES.get(r['id'], auto_setting)
    assert setting in CANNY_SETTINGS, f'Invalid threshold setting for {r["id"]}'
    rank = CONTOUR_RANK_OVERRIDES.get(r['id'], 0)
    assert 0 <= rank < len(r['candidates'][setting]), f'No such contour rank for {r["id"]}'
    r['best_setting'] = setting
    r['boundary'] = r['candidates'][setting][rank]
    r['choice_source'] = 'manual' if r['id'] in THRESHOLD_OVERRIDES or r['id'] in CONTOUR_RANK_OVERRIDES else 'automatic proposal'
    print(f"{r['id']}: {setting[0]}–{setting[1]} ({r['choice_source']}); "
          f"area {r['boundary']['area']} pixels; perimeter {r['boundary']['perimeter']:.1f} pixels")

## 9. Final required five-stage figure and measurements

We now make **Original → Grayscale → Gaussian Filter → Canny → Lesion Boundary** for each of five images. The area is the number of white pixels in the filled mask. The perimeter is `cv2.arcLength` of the chosen contour. Both are approximate, depend on the outline, and are measured at the image's original pixel dimensions.

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(16, 17))
for i, r in enumerate(records):
    setting = r['best_setting']
    overlay = r['rgb'].copy()
    cv2.drawContours(overlay, [r['boundary']['contour']], -1, (255, 255, 0), 3)
    views = [r['rgb'], r['gray'], r['gaussian'], r['canny'][setting], overlay]
    titles = ['Original', 'Grayscale', 'Gaussian filter',
              f'Canny {setting[0]}–{setting[1]}',
              f"Boundary | A={r['boundary']['area']} px² | P={r['boundary']['perimeter']:.1f} px"]
    for ax, im, title in zip(axes[i], views, titles):
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
        ax.set_title(title, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'final_five_stage_visualization.png', dpi=170); plt.show()

rows = [{'Image': f'Image {i+1} ({r["id"]})', 'Best Filter': 'Gaussian (5×5)',
         'Edge Method': f'Canny {r["best_setting"][0]}–{r["best_setting"][1]}',
         'Area (pixels)': r['boundary']['area'],
         'Perimeter (pixels)': round(r['boundary']['perimeter'], 1),
         'Selection': r['choice_source']} for i, r in enumerate(records)]
results = pd.DataFrame(rows)
display(results)
results.to_csv(OUT/'five_image_results.csv', index=False)

## 10. Compare Sobel and Canny with four input choices

This completes the assignment's eight-row comparison: Original, Average, Gaussian and Median, each with Sobel or Canny. We add a **small reproducible synthetic noise perturbation** solely to measure how much the edge map changes. `Noise Handling` reports overlap with the clean edge map (higher = more stable); **stability is not segmentation accuracy**. `Boundary Detection` counts how many of five photos produce a *plausible candidate*, not medically verified success. Edge density gives a rough indication of how many pixels were marked as edges; **more is not automatically better**. Inspect the eight-way figure before trusting the final column.

In [ ]:
def apply_filter(gray, name):
    if name == 'Original': return gray
    if name == 'Average': return cv2.blur(gray, (5, 5))
    if name == 'Gaussian': return cv2.GaussianBlur(gray, (5, 5), 0)
    if name == 'Median': return cv2.medianBlur(gray, 5)
    raise ValueError(name)

def detect_edges(gray, method):
    if method == 'Canny': return cv2.Canny(gray, 100, 200)
    sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    return np.uint8(cv2.magnitude(sx, sy) >= 100) * 255

def binary_f1(a, b):
    x, y = a > 0, b > 0
    tp = np.logical_and(x, y).sum()
    return float(2*tp/(x.sum()+y.sum())) if x.any() or y.any() else 1.0

METHODS = [(f, d) for f in ('Original', 'Average', 'Gaussian', 'Median')
          for d in ('Sobel', 'Canny')]
rng = np.random.default_rng(SEED)
comparison_rows = []
for filt, detector in METHODS:
    stability, density, candidates_found = [], [], 0
    for r in records:
        clean = detect_edges(apply_filter(r['gray'], filt), detector)
        noisy = np.uint8(np.clip(r['gray'].astype(np.float32) +
                                 rng.normal(0, 12, r['gray'].shape), 0, 255))
        noisy_edge = detect_edges(apply_filter(noisy, filt), detector)
        stability.append(binary_f1(clean, noisy_edge))
        density.append(float(np.mean(clean > 0)))
        candidate_list, _ = find_boundary(clean, r['gray'])
        candidates_found += bool(candidate_list)
    mean_stability, mean_density = float(np.mean(stability)), float(np.mean(density))
    # 'Promising' means these proxies look reasonable, not that an expert validated the boundary.
    proxy = 'Promising — review outlines' if mean_stability >= .50 and candidates_found >= 4 else 'Needs visual review'
    comparison_rows.append({
        'Method': f'{filt} + {detector}',
        'Noise Handling': f'F1 stability {mean_stability:.2f}',
        'Edge Quality': f'{mean_density*100:.1f}% edge pixels; inspect',
        'Boundary Detection': f'{candidates_found}/5 candidates; inspect',
        'Overall Performance': proxy,
        '_stability': mean_stability, '_density': mean_density,
        '_candidates': candidates_found})

comparison = pd.DataFrame(comparison_rows)
display(comparison[['Method', 'Noise Handling', 'Edge Quality',
                    'Boundary Detection', 'Overall Performance']])
comparison.drop(columns=['_stability','_density','_candidates']).to_csv(
    OUT/'eight_method_comparison.csv', index=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample = records[0]
for ax, (filt, detector) in zip(axes.flat, METHODS):
    ax.imshow(detect_edges(apply_filter(sample['gray'], filt), detector), cmap='gray')
    ax.set_title(f'{filt} + {detector}'); ax.axis('off')
plt.suptitle(f'Eight-method visual check | {sample["id"]}')
plt.tight_layout(); plt.savefig(OUT/'eight_method_visual_comparison.png', dpi=160); plt.show()

## 11. Answer the six written questions

1. **Why Gaussian before Canny?** Gaussian smoothing reduces tiny brightness changes from noise. Canny then finds fewer false edges. Too much blur can erase a faint lesion boundary.
2. **How did thresholds affect the result?** A low pair such as 50–100 normally preserves more detail but also more distracting edges. A high pair such as 150–250 normally gives fewer, potentially broken edges. Confirm this with your own five images above.
3. **Which threshold was best?** Read the *Edge Method* column in your generated five-image table. Different photos may require different settings. State the actual selected pair for **each** photo and mention that the traced contour was visually inspected.
4. **Why are edges useful?** A lesion can differ in brightness from surrounding skin, so its margin may appear as a curve of strong pixel changes. Edges help locate that curve.
5. **What problems occurred?** Hair, skin lines, faint/irregular lesion borders, shadows and small disconnected edges can cause false or incomplete contours. Point to the issues actually visible in your five examples.
6. **How could it improve?** Better hair removal, controlled lighting, local contrast adjustment, stronger contour selection, and evaluation against expert masks (for example overlap/IoU and Dice) would help. The pixel area would need a physical scale for mm².

**Conclusion template:** “For our five HAM10000 images, Gaussian filtering and Canny gave the outlined examples shown above. The measured areas and perimeters are estimates from the selected masks. The eight-method table reports edge stability and candidate coverage, which are proxies. We cannot claim lesion-segmentation accuracy without expert reference masks.”

## 12. Download the result files

Run this last cell after inspecting your five boundaries and any overrides. It bundles the figures and two completed tables for submission. The notebook itself should also be downloaded from **File → Download → Download .ipynb** in Colab.

In [ ]:
archive_path = Path('/content/lesion_boundary_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(OUT.iterdir()):
        if file_path.is_file(): archive.write(file_path, file_path.name)
print('Files included:')
for p in sorted(OUT.iterdir()): print(' ', p.name)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Outside Colab; archive:', archive_path)